<a href="https://colab.research.google.com/github/MaSBroEarl/Task.Text-generation/blob/main/LoRA%2BLLMasJudge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Импорт библиотек и загрузка модели:

Qwen/Qwen2.5-1.5B-Instruct

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Загрузка модели...")

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("✅ Модель загружена!")

Загрузка модели...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Модель загружена!


In [2]:
# Берём один диалог
df = pd.read_csv('/content/train.csv')
test_dialogue = df.iloc[0]['dialogue']
test_summary = df.iloc[0]['summary']

prompt = f"""<|im_start|>user
Summarize this dialogue concisely:

{test_dialogue}
<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
prediction = response.split("<|im_start|>assistant")[-1].strip()

print("Референс:", test_summary)
print("Предсказание:", prediction)

Референс: Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.
Предсказание: user
Summarize this dialogue concisely:

#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I 

Импорт библиотек для LoRA

In [3]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from transformers import BitsAndBytesConfig

print("✅ LoRA библиотеки импортированы")

✅ LoRA библиотеки импортированы


Подготовка данных для LoRA

In [4]:
df = pd.read_csv('/content/train.csv')
df_train = df.sample(500, random_state=42)
df_eval = df.sample(100, random_state=43)

def format_example(row):
    return f"""<|im_start|>user
Summarize this dialogue concisely:

{row['dialogue']}
<|im_end|>
<|im_start|>assistant
{row['summary']}
<|im_end|>"""

train_dataset = Dataset.from_dict({"text": [format_example(row) for _, row in df_train.iterrows()]})
eval_dataset = Dataset.from_dict({"text": [format_example(row) for _, row in df_eval.iterrows()]})

print("✅ Данные готовы")
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

✅ Данные готовы
Train: 500, Eval: 100


In [5]:
def tokenize_function(examples):
    result = tokenizer(examples["text"], truncation=True, padding=False, max_length=512)
    result["labels"] = result["input_ids"].copy()  # ВАЖНО: добавляем labels
    return result

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print("✅ Токенизация готова")
print(f"Train: {len(tokenized_train)}, Eval: {len(tokenized_eval)}")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Токенизация готова
Train: 500, Eval: 100


In [7]:
!pip install torchao==0.16.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.8 MB/s eta 0:00:00


In [8]:
# Подготовка модели
model = prepare_model_for_kbit_training(model)

# Конфигурация LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Аргументы обучения
training_args = TrainingArguments(
    output_dir="./qwen-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="no",
    learning_rate=2e-4,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

print("🚀 Обучение...")
trainer.train()
print("✅ Обучение завершено!")

# Сохраняем модель
model.save_pretrained("./qwen-lora-final")
tokenizer.save_pretrained("./qwen-lora-final")
print("✅ Модель сохранена!")

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
🚀 Обучение...


Step,Training Loss
10,2.190719
20,2.203236
30,2.103640
40,2.006020
50,1.887537
60,1.674848
70,1.626667
80,1.689618
90,1.653025
100,1.682308


✅ Обучение завершено!
✅ Модель сохранена!


Делаем инференс на тестовых примерах

In [9]:
# Берём 20 тестовых примеров
test_df = df.sample(20, random_state=123)

predictions = []
references = []

for _, row in test_df.iterrows():
    prompt = f"""<|im_start|>user
Summarize this dialogue concisely:

{row['dialogue']}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = response.split("<|im_start|>assistant")[-1].strip()

    predictions.append(pred)
    references.append(row['summary'])

# Смотрим результаты
for i in range(5):
    print(f"\n--- Пример {i+1} ---")
    print(f"Референс: {references[i]}")
    print(f"Предсказание: {predictions[i]}")
    print("-"*50)


--- Пример 1 ---
Референс: Steven wakes up and thinks he's late for work. Julia tells him it's Sunday and asks him to get more sleep after breakfast.
Предсказание: user
Summarize this dialogue concisely:

#Person1#: Julia, what time is it?
#Person2#: Eight o'clock. It's time for you to get up and have breakfast.
#Person1#: Oh, my God! I'm going to be late! I have no time to have breakfast now. ( Hurry on his clothes. )
#Person2#: You won't go to work today, Steven, It's Sunday. Come and have breakfast now.
#Person1#: Oh, I have a poor memory now. I haven't had enough sleep lately. I had a bad dream just now.
#Person2#: You have been too tired recently, darling. That's why I didn't wake you up this morning. After breakfast, you can go to sleep again.
#Person1#: Yes. I really need to have a good rest.

assistant
Steven gets up early in the morning because he has a bad dream, so #Person2# wakes him up with breakfast.
--------------------------------------------------

--- Пример 2 ---
Ре

In [11]:
!pip install rouge-score -q

  Preparing metadata (setup.py) ... done


In [12]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoothie = SmoothingFunction().method4

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
bleu_scores = []

for ref, pred in zip(references, predictions):
    scores = scorer.score(ref, pred)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

    bleu = sentence_bleu([ref.split()], pred.split(), smoothing_function=smoothie)
    bleu_scores.append(bleu)

print("="*50)
print("МЕТРИКИ (на 20 примерах)")
print("="*50)
print(f"ROUGE-1: {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2: {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L: {np.mean(rougeL_scores):.4f}")
print(f"BLEU:    {np.mean(bleu_scores):.4f}")

МЕТРИКИ (на 20 примерах)
ROUGE-1: 0.2123
ROUGE-2: 0.0888
ROUGE-L: 0.1571
BLEU:    0.0275


LLM как судья
Groq

In [13]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.6 MB/s eta 0:00:00


In [15]:
from groq import Groq

# Вставь свой API ключ
client = Groq(api_key="ТОКЕН")

def judge_summary(dialogue, reference, prediction):
    prompt = f"""You are an expert evaluator of text summarization.

Dialogue:
{dialogue}

Reference Summary:
{reference}

Candidate Summary:
{prediction}

Rate the candidate summary on a scale of 1-5 based on:
1. Relevance (does it capture key points?)
2. Conciseness (is it concise?)
3. Fluency (is it readable?)

Also provide a brief justification.

Output format:
Relevance: [score]
Conciseness: [score]
Fluency: [score]
Justification: [text]
"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300,
    )
    return response.choices[0].message.content

# Проверяем на 5 примерах
for i in range(5):
    print(f"\n--- Пример {i+1} ---")
    print(judge_summary(
        test_df.iloc[i]['dialogue'][:500],
        references[i],
        predictions[i]
    ))
    print("="*50)


--- Пример 1 ---
Relevance: 3
Conciseness: 4
Fluency: 5

Justification:
- Relevance: The candidate summary captures the main points of the dialogue, but it doesn't fully convey the context. It mentions Steven getting up early because of a bad dream, which is partially correct, but it doesn't mention that it's Sunday and that Julia is trying to get him to have breakfast and rest. 
- Conciseness: The candidate summary is concise and to the point, but it could be improved by removing the phrase "because he has a bad dream," which is not entirely accurate. 
- Fluency: The candidate summary is well-written and easy to read.

--- Пример 2 ---
Relevance: 4
Conciseness: 4
Fluency: 5

Justification:
The candidate summary captures the main points of the dialogue, including #Person1#'s stress about college and scholarships, and #Person2#'s request for help with English. However, it misses the part where #Person2# initially talks about their ideal work situation, which is not directly related to 